# SEMIR LiTS Reproduction — corrected v3

**Goal:** reproduce the LiTS tumor-vs-rest result reported for SEMIR.

This version corrects the reproduction-critical issues in `semir_lits_v2.ipynb`:

1. Replaces the quick `merge_distance` sweep with a real few-shot search over SEMIR-style parameters: `ψ`, `α`, `β_min`, `β_max`, `m_min`, `m_max`.
2. Uses boundary Dice on a few-shot subset as the search objective, while logging oracle Dice, tumor deletion, and supernode count to catch degenerate graph minors.
3. Adds true per-supernode intensity standard deviation.
4. Represents dominant axis as the three principal eigenvector components instead of a dominant eigenvalue or eigenvector norm.
5. Builds PyG graphs from the selected `Θopt`, not hard-coded constants.
6. Selects checkpoints using lifted voxel-level validation Dice, not supernode Dice.

**Paper target:** LiTS tumor Dice ≈ `0.891 ± 0.007`, with LiTS graph size around `1,075 ± 297` supernodes.


## 1. Setup

In [ ]:
import numpy as np
import os, re, time, json, math, random
import fastloops

DATA_ROOT = "/scratch/ud3d4/acm_data/Data"

# Default HU window from the SWOG notes. You can also test (0, 200) in the diagnostic cell.
HU_MIN, HU_MAX = -50, 250

# These are overwritten by the few-shot search in Section 4.
MERGE_DIST = 8       # ψ
CUT_DIST = 51        # α
DELETE_SMALL = 5     # β_min
DELETE_LARGE_FRAC = 0.80  # β_max = int(n_vox ** DELETE_LARGE_FRAC)
VALUE_MIN = 13       # m_min
VALUE_MAX = 242      # m_max
OVERLAP_THRESHOLD = 0.50

np.random.seed(42)
random.seed(42)

print("fastloops loaded")
print("Initial params will be overwritten by few-shot search:")
print(f"  HU=[{HU_MIN},{HU_MAX}]  ψ={MERGE_DIST}  α={CUT_DIST}")
print(f"  β_min={DELETE_SMALL}  β_max=n_vox**{DELETE_LARGE_FRAC}  m=[{VALUE_MIN},{VALUE_MAX}]")


## 2. Discover LiTS Volumes

In [ ]:
def discover_volumes():
    """Find all LiTS volumes that have tumor (label=2)."""
    ct_dir = os.path.join(DATA_ROOT, "ct")
    ids = []
    for f in sorted(os.listdir(ct_dir)):
        m = re.match(r"volume-(\d+)\.npy", f)
        if m:
            vid = int(m.group(1))
            seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy"))
            if (seg == 2).sum() > 0:
                ids.append(vid)
    return sorted(ids)

all_vids = discover_volumes()
print(f"Found {len(all_vids)} LiTS volumes with tumor")

# Split: 70/15/15
np.random.seed(42)
perm = np.random.permutation(len(all_vids))
n_train = int(0.7 * len(all_vids))
n_val = int(0.15 * len(all_vids))
train_ids = sorted([all_vids[i] for i in perm[:n_train]])
val_ids = sorted([all_vids[i] for i in perm[n_train:n_train + n_val]])
test_ids = sorted([all_vids[i] for i in perm[n_train + n_val:]])
ordered = train_ids + val_ids + test_ids
print(f"Split: {len(train_ids)} train / {len(val_ids)} val / {len(test_ids)} test")

## 3. Rust Contraction + Deletion (Single Call)

Luke's `merge_and_cut` does all three graph minor operations in one interleaved pass:
1. **Edge contraction** — seed-based flood fill, compares each voxel to region's canonical seed
2. **Node deletion** — if region is too small/large or wrong intensity, voxels get **unflagged** so neighboring seeds re-absorb them
3. **Edge deletion** — marks edges between regions with intensity diff >= cut_distance

The re-absorption mechanism (line 119 of lib.rs) is critical — deleted voxels become available for future seeds.

In [ ]:
def load_and_convert(vid, hu_min=None, hu_max=None):
    """Load raw CT and convert to uint8 with the requested HU window."""
    hu_min = HU_MIN if hu_min is None else hu_min
    hu_max = HU_MAX if hu_max is None else hu_max
    ct = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
    seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
    ct_u8 = np.clip(ct, hu_min, hu_max)
    ct_u8 = ((ct_u8 - hu_min) / (hu_max - hu_min) * 255).round().astype(np.uint8)
    ct_u8 = np.ascontiguousarray(ct_u8[..., np.newaxis])  # (D, H, W, 1)
    return ct, seg, ct_u8


def oracle_dice(labels_np, seg):
    """Quality ceiling of the graph minor.

    A perfect supernode classifier marks every surviving supernode that contains
    any tumor voxel as foreground. If this score is low, the coarsener has already
    destroyed the segmentation problem.
    """
    flat = labels_np.ravel()
    gt = (seg.ravel() == 2).astype(np.float64)
    gt_total = int(gt.sum())
    valid = flat >= 0
    if gt_total == 0 or not valid.any():
        return 0.0, 0, 100.0 if gt_total > 0 else 0.0
    max_id = int(flat[valid].max())
    tc = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
    tumor_sids = np.where(tc > 0)[0]
    lut = np.zeros(max_id + 1, dtype=np.int32)
    lut[tumor_sids] = 1
    pred = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
    gt_mask = seg == 2
    inter = int((pred & gt_mask).sum())
    dice = 2.0 * inter / (pred.sum() + gt_mask.sum() + 1e-8)
    deleted_tumor = int(gt[~valid].sum())
    deleted_pct = deleted_tumor / max(gt_total, 1) * 100.0
    return dice, len(tumor_sids), deleted_pct


def compute_intensity_std(labels_np, ct_u8):
    """Per-supernode intensity std in [0,1], indexed by supernode ID."""
    flat = labels_np.ravel()
    valid = flat >= 0
    if not valid.any():
        return np.array([], dtype=np.float32)
    max_id = int(flat[valid].max())
    vals = ct_u8[..., 0].ravel().astype(np.float64) / 255.0
    counts = np.bincount(flat[valid], minlength=max_id + 1).astype(np.float64)
    sums = np.bincount(flat[valid], weights=vals[valid], minlength=max_id + 1)
    sq_sums = np.bincount(flat[valid], weights=vals[valid] ** 2, minlength=max_id + 1)
    mean = sums / np.maximum(counts, 1.0)
    var = sq_sums / np.maximum(counts, 1.0) - mean ** 2
    return np.sqrt(np.maximum(var, 0.0)).astype(np.float32)


def boundary_mask_6conn(mask):
    """6-connected binary boundary without scipy dependency."""
    mask = mask.astype(bool)
    b = np.zeros_like(mask, dtype=bool)
    for axis in range(3):
        lo = [slice(None)] * 3
        hi = [slice(None)] * 3
        lo[axis] = slice(0, -1)
        hi[axis] = slice(1, None)
        diff = mask[tuple(lo)] != mask[tuple(hi)]
        b[tuple(lo)] |= diff
        b[tuple(hi)] |= diff
    return b & mask


def supernode_boundary_mask(labels_np):
    """Voxels adjacent to a different surviving supernode."""
    valid = labels_np >= 0
    b = np.zeros_like(labels_np, dtype=bool)
    for axis in range(3):
        lo = [slice(None)] * 3
        hi = [slice(None)] * 3
        lo[axis] = slice(0, -1)
        hi[axis] = slice(1, None)
        a = labels_np[tuple(lo)]
        c = labels_np[tuple(hi)]
        v = valid[tuple(lo)] & valid[tuple(hi)]
        diff = (a != c) & v
        b[tuple(lo)] |= diff
        b[tuple(hi)] |= diff
    return b & valid


def boundary_dice(labels_np, seg, target_label=2):
    """SEMIR few-shot objective: DSC(supernode boundaries, target GT boundary)."""
    gt_b = boundary_mask_6conn(seg == target_label)
    sn_b = supernode_boundary_mask(labels_np)
    denom = gt_b.sum() + sn_b.sum()
    if denom == 0:
        return 0.0
    return 2.0 * int((gt_b & sn_b).sum()) / (denom + 1e-8)


def run_minor(ct_u8, n_vox, params):
    """One wrapper around the Rust coarsener."""
    beta_max = int(n_vox ** float(params["beta_max_frac"]))
    return fastloops.merge_and_cut(
        ct_u8,
        merge_distance=int(params["psi"]),
        cut_distance=int(params["alpha"]),
        delete_small_node_max_size=int(params["beta_min"]),
        delete_large_node_min_size=beta_max,
        delete_value_min=int(params["m_min"]),
        delete_value_max=int(params["m_max"]),
        connectivity="faces",
    )

print("Helper functions defined.")


In [ ]:
# Run graph minor construction on all volumes
minors = {}
oracle_results = []

for vid in ordered:
    ct_raw, seg, ct_u8 = load_and_convert(vid)
    n_vox = ct_raw.size

    t0 = time.time()
    # Single call — Rust handles contraction + deletion + edge cutting
    raw_nf, raw_ei, raw_ef, raw_labels, raw_adj = fastloops.merge_and_cut(
        ct_u8,
        merge_distance=MERGE_DIST,
        cut_distance=CUT_DIST,
        delete_small_node_max_size=DELETE_SMALL,
        delete_large_node_min_size=int(n_vox ** 0.8),
        delete_value_min=VALUE_MIN,
        delete_value_max=VALUE_MAX,
        connectivity="faces",
    )
    dt = time.time() - t0

    labels_np = np.asarray(raw_labels)
    n_sn = raw_nf.shape[0]
    n_edges = raw_ei.shape[1]

    # Oracle Dice (quality ceiling)
    od, t_sn, del_pct = oracle_dice(labels_np, seg)

    # Convert labels for downstream: -1 → 0 (deleted), 0-based → 1-based
    labels_out = labels_np.copy()
    labels_out[labels_np >= 0] += 1
    labels_out[labels_np < 0] = 0

    # Build adjacency dict for feature extraction
    full_adjacency = {}
    if n_edges > 0:
        a_ids = raw_ei[0].astype(int) + 1
        b_ids = raw_ei[1].astype(int) + 1
        ef_np = raw_ef.astype(np.float64)
        for idx in range(n_edges):
            key = (int(min(a_ids[idx], b_ids[idx])), int(max(a_ids[idx], b_ids[idx])))
            mean_contrast = ef_np[idx, 1] / max(ef_np[idx, 0], 1) / 255.0
            full_adjacency[key] = float(mean_contrast)

    minors[vid] = {
        "labels": labels_out, "n_supernodes": n_sn,
        "adjacency": full_adjacency, "full_adjacency": full_adjacency,
        "stats": {"n_voxels": n_vox, "n_supernodes": n_sn, "n_edges": n_edges,
                  "compression_ratio": n_vox / max(n_sn, 1), "time_s": round(dt, 2)},
    }
    oracle_results.append({"vid": vid, "oracle": round(od, 4), "n_sn": n_sn,
                           "n_edges": n_edges, "del_pct": round(del_pct, 1), "t_sn": t_sn})

    print(f"vol-{vid}: {n_vox:>10,} -> {n_sn:>6,} SN  {n_edges:,} edges  "
          f"oracle={od:.4f}  del={del_pct:.1f}%  {dt:.1f}s")

mean_oracle = np.mean([r["oracle"] for r in oracle_results])
mean_sn = np.mean([r["n_sn"] for r in oracle_results])
print(f"\nMean oracle: {mean_oracle:.4f}  Mean SN: {mean_sn:.0f}")
print(f"Paper target: oracle ~1.0, ~1075 SN")

## 3b. Oracle Dice — HU Window Comparison (The 0.91 Finding)

The oracle Dice depends heavily on the HU window. We compare:
- **HU [0, 200]** (liver window) — concentrates tumor-liver contrast in uint8 space
- **HU [-50, 250]** (Luke's window) — wider range, dilutes contrast

Both use the same Rust `merge_and_cut` with **no deletion** (`delete_small=0`).
This isolates the effect of the HU window on contraction quality.

In [ ]:
# Compare oracle Dice with two HU windows — NO deletion
# This shows where the 0.91 oracle came from and why Luke's window gives 0.04

hu_configs = [
    (0, 200, "HU [0, 200] liver window"),
    (-50, 250, "HU [-50, 250] Luke's window"),
]

test_vids = ordered[:20]  # first 20 for speed

for hu_min, hu_max, label in hu_configs:
    oracles, sns = [], []
    for vid in test_vids:
        ct_raw = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
        seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
        n_vox = ct_raw.size

        ct_u8 = np.clip(ct_raw, hu_min, hu_max)
        ct_u8 = ((ct_u8 - hu_min) / (hu_max - hu_min) * 255).round().astype(np.uint8)
        ct_u8 = np.ascontiguousarray(ct_u8[..., np.newaxis])

        # NO deletion — contraction only
        nf, ei, ef, labels, adj = fastloops.merge_and_cut(
            ct_u8,
            merge_distance=MERGE_DIST,
            cut_distance=CUT_DIST,
            delete_small_node_max_size=0,
            delete_large_node_min_size=n_vox,
            delete_value_min=0,
            delete_value_max=255,
            connectivity="faces",
        )
        labels_np = np.asarray(labels)
        od, t_sn, del_pct = oracle_dice(labels_np, seg)
        oracles.append(od)
        sns.append(nf.shape[0])

    print(f"{label}:")
    print(f"  Oracle Dice: {np.mean(oracles):.4f} +/- {np.std(oracles):.4f}  "
          f"(min={np.min(oracles):.4f}, max={np.max(oracles):.4f})")
    print(f"  Supernodes:  {np.mean(sns):,.0f}\n")

print("The 0.91 oracle came from HU [0,200] with merge_dist=5 (even tighter).")
print("Luke's HU [-50,250] dilutes tumor-liver contrast in uint8 space,")
print("causing impure boundary supernodes that kill oracle via false positives.")


## 4. Few-shot SEMIR parameter search

The paper does **not** use one fixed hand-picked coarsening setting. It selects minor parameters on 5–20 labeled cases with a boundary-alignment objective.

This cell searches over:

- `ψ`: contraction threshold / `merge_distance`
- `α`: edge deletion threshold / `cut_distance`, constrained to be above `ψ`
- `β_min`: small-node deletion threshold
- `β_max`: large-node deletion threshold, expressed as `n_vox ** beta_max_frac`
- `m_min`, `m_max`: intensity deletion bounds

The primary score is mean boundary Dice. The diagnostics matter just as much: if oracle Dice is low or tumor deletion is high, the GNN cannot recover the tumor mask.


In [ ]:
# -------------------------
# Few-shot parameter search
# -------------------------
N_FEW = min(5, len(train_ids))
N_RANDOM = 400          # increase to 400-1000 for a slower, stronger search
TOP_K_PRINT = 12
few_vids = train_ids[:N_FEW]
print(f"Few-shot volumes: {few_vids}")

few_data = []
for vid in few_vids:
    ct_raw, seg, ct_u8 = load_and_convert(vid)
    few_data.append({"vid": vid, "seg": seg, "ct_u8": ct_u8, "n_vox": ct_raw.size})

# Seed candidates include the prior hand setting plus progressively more aggressive contraction.
seed_candidates = [
    dict(psi=5,  alpha=35,  beta_min=0,  beta_max_frac=0.90, m_min=0,  m_max=255),
    dict(psi=8,  alpha=51,  beta_min=5,  beta_max_frac=0.80, m_min=13, m_max=242),
    dict(psi=12, alpha=70,  beta_min=3,  beta_max_frac=0.86, m_min=5,  m_max=250),
    dict(psi=20, alpha=100, beta_min=1,  beta_max_frac=0.88, m_min=0,  m_max=255),
    dict(psi=30, alpha=150, beta_min=0,  beta_max_frac=0.90, m_min=0,  m_max=255),
]

rng = np.random.default_rng(42)
random_candidates = []
for _ in range(N_RANDOM):
    psi = int(rng.choice([4,5,6,8,10,12,15,18,20,24,28,32,36,42,50]))
    alpha_min = max(psi + 8, int(psi * 2.0))
    alpha = int(rng.integers(alpha_min, 256))
    beta_min = int(rng.choice([0,1,2,3,5,8,13,21,34,55,89,144,233]))
    beta_max_frac = float(rng.choice([0.76,0.80,0.84,0.88,0.92,0.96,1.00]))
    # Intensity deletion can easily delete tumor; include wide settings often.
    if rng.random() < 0.45:
        m_min, m_max = 0, 255
    else:
        m_min = int(rng.choice([0,3,5,8,13,21,34]))
        m_max = int(rng.choice([220,230,242,250,255]))
        if m_min >= m_max:
            m_min, m_max = 0, 255
    random_candidates.append(dict(psi=psi, alpha=alpha, beta_min=beta_min,
                                  beta_max_frac=beta_max_frac, m_min=m_min, m_max=m_max))

# De-duplicate candidates.
seen = set()
candidates = []
for c in seed_candidates + random_candidates:
    key = tuple(c[k] for k in ["psi", "alpha", "beta_min", "beta_max_frac", "m_min", "m_max"])
    if key not in seen:
        seen.add(key)
        candidates.append(c)

print(f"Evaluating {len(candidates)} candidates on {N_FEW} cases...")
rows = []
t0 = time.time()
for idx, params in enumerate(candidates, start=1):
    bds, ods, sns, dels, tumor_sns = [], [], [], [], []
    ok = True
    for item in few_data:
        try:
            nf, ei, ef, labels, adj = run_minor(item["ct_u8"], item["n_vox"], params)
            labels_np = np.asarray(labels)
            bd = boundary_dice(labels_np, item["seg"])
            od, t_sn, del_pct = oracle_dice(labels_np, item["seg"])
            bds.append(bd); ods.append(od); sns.append(nf.shape[0]); dels.append(del_pct); tumor_sns.append(t_sn)
        except Exception as e:
            ok = False
            break
    if not ok:
        continue
    row = dict(params)
    row.update(boundary=float(np.mean(bds)), oracle=float(np.mean(ods)),
               sn=float(np.mean(sns)), del_pct=float(np.mean(dels)),
               tumor_sn=float(np.mean(tumor_sns)))
    # Selection score: paper objective (boundary) with guardrails against degenerate minors.
    # If you want strict paper-only selection, replace this line with: row["score"] = row["boundary"]
    sn_penalty = 0.00000002 * max(row["sn"] - 5000, 0)  # soft penalty above 5k nodes
    deletion_penalty = 0.02 * max(row["del_pct"] - 2.0, 0)
    oracle_penalty = 0.25 * max(0.65 - row["oracle"], 0)
    row["score"] = row["boundary"] - sn_penalty - deletion_penalty - oracle_penalty
    rows.append(row)
    if idx % 25 == 0:
        print(f"  {idx:4d}/{len(candidates)} candidates, elapsed={time.time()-t0:.1f}s")

rows = sorted(rows, key=lambda r: r["score"], reverse=True)
print("\nTop candidates:")
print(f"{'rank':>4s} {'score':>8s} {'bd':>7s} {'oracle':>7s} {'SN':>8s} {'del%':>6s}  params")
for rank, r in enumerate(rows[:TOP_K_PRINT], start=1):
    ptxt = f"ψ={r['psi']} α={r['alpha']} βmin={r['beta_min']} βmax=n^{r['beta_max_frac']:.2f} m=[{r['m_min']},{r['m_max']}]"
    print(f"{rank:4d} {r['score']:8.4f} {r['boundary']:7.4f} {r['oracle']:7.4f} {r['sn']:8.0f} {r['del_pct']:5.1f}%  {ptxt}")

if not rows:
    raise RuntimeError("No valid SEMIR candidates were evaluated.")

BEST = rows[0]
MERGE_DIST = int(BEST["psi"])
CUT_DIST = int(BEST["alpha"])
DELETE_SMALL = int(BEST["beta_min"])
DELETE_LARGE_FRAC = float(BEST["beta_max_frac"])
VALUE_MIN = int(BEST["m_min"])
VALUE_MAX = int(BEST["m_max"])

print("\nSelected Θopt:")
print(json.dumps({k: BEST[k] for k in ["psi", "alpha", "beta_min", "beta_max_frac", "m_min", "m_max", "boundary", "oracle", "sn", "del_pct"]}, indent=2))

if BEST["oracle"] < 0.80:
    print("\nWARNING: Few-shot oracle is still below 0.80. Train the GINE only as a diagnostic; the graph minor is still likely the bottleneck.")
if BEST["sn"] > 10000:
    print("WARNING: Mean supernodes are still high. Increase N_RANDOM or broaden ψ/β search if runtime permits.")


## 5. Feature extraction and graph construction

This version uses:

- log volume
- log boundary/surface
- compactness
- elongation
- **principal-axis components** `axis_x, axis_y, axis_z`
- mean intensity
- true intensity std

That yields 9 node features for a single-channel CT volume. The paper lists dominant axis as one descriptor, but the actual axis is a 3D vector, so using its components is the least lossy implementation.


In [ ]:
# Luke's feature extraction functions (from rust_crate.ipynb)

def _layout(C):
    return dict(area=0, s=[1, 2, 3],
                cov=[(4, 0, 0), (5, 1, 1), (6, 2, 2), (7, 0, 1), (8, 0, 2), (9, 1, 2)],
                chan0=10, boundary=10 + C + 6, D=3)


def node_invariants(node_feats, C=1, eps=1e-6):
    """Extract scale/orientation-invariant node features from Rust output."""
    f = node_feats.astype(np.float64)
    L = _layout(C)
    D = L["D"]
    N = f.shape[0]
    V = f[:, L["area"]]
    Vsafe = np.maximum(V, 1.0)
    mean_coord = np.stack([f[:, c] for c in L["s"]], axis=1) / Vsafe[:, None]

    cov = np.zeros((N, D, D))
    for col, i, j in L["cov"]:
        cij = f[:, col] / Vsafe - mean_coord[:, i] * mean_coord[:, j]
        cov[:, i, j] = cij; cov[:, j, i] = cij

    w = np.linalg.eigvalsh(cov)
    w = np.clip(w, 0.0, None)
    _, vec = np.linalg.eigh(cov)
    principal = vec[..., -1]
    trace = w.sum(axis=1)
    degenerate = trace < eps

    denom = w[:, 2] + eps
    shape = np.stack([(w[:, 2] - w[:, 1]) / denom,
                      (w[:, 1] - w[:, 0]) / denom,
                      w[:, 0] / denom], axis=1)
    shape[degenerate] = 0.0
    line_like = np.where(degenerate, 0.0, shape[:, 0])

    chan = f[:, L["chan0"]:L["chan0"] + C] / Vsafe[:, None] / 255.0
    compactness = f[:, L["boundary"]] / np.power(Vsafe, (D - 1.0) / D)
    elongation = np.where(w[:, 0] > eps, w[:, 2] / (w[:, 0] + eps), 1.0)
    elongation = np.clip(elongation, 1.0, 100.0)

    return dict(V=V, surface=f[:, L["boundary"]], centroid=mean_coord,
                eig=w, shape=shape, line_like=line_like, principal=principal,
                chan=chan, compactness=compactness, elongation=elongation,
                log_size=np.log(Vsafe))


def edge_invariants(node_feats, edge_index, edge_feats, C=1, eps=1e-6):
    """Extract scale-invariant edge features from Rust output."""
    inv = node_invariants(node_feats, C, eps)
    a = edge_index[0].astype(np.int64)
    b = edge_index[1].astype(np.int64)
    ef = edge_feats.astype(np.float64)
    blsafe = np.maximum(ef[:, 0], 1.0)

    size_contrast = np.abs(inv["V"][a] - inv["V"][b]) / (inv["V"][a] + inv["V"][b] + eps)
    bfrac_a = ef[:, 0] / (inv["surface"][a] + eps)
    bfrac_b = ef[:, 0] / (inv["surface"][b] + eps)
    mean_contrast = np.abs(inv["chan"][a] - inv["chan"][b])
    shape_dissim = np.abs(inv["shape"][a] - inv["shape"][b])
    axis_align = (np.abs(np.sum(inv["principal"][a] * inv["principal"][b], axis=1))
                  * np.minimum(inv["line_like"][a], inv["line_like"][b]))
    bcontrast = (ef[:, 1] / blsafe) / 255.0
    cut_frac = ef[:, 3] / blsafe

    cols = [size_contrast[:, None], bfrac_a[:, None], bfrac_b[:, None],
            mean_contrast if mean_contrast.ndim > 1 else mean_contrast[:, None],
            shape_dissim, axis_align[:, None], bcontrast[:, None], cut_frac[:, None]]
    return np.concatenate(cols, axis=1).astype(np.float32)

In [ ]:
import torch
from torch_geometric.data import Data


def build_pyg_graph(raw_nf, raw_ei, raw_ef, labels_np, seg, ct_u8, C=1, overlap_threshold=OVERLAP_THRESHOLD):
    """Build a PyG graph from the Rust minor output.

    Labels are assigned by majority/overlap threshold on each supernode. The final
    metric is still lifted voxel Dice, so this threshold should be swept if oracle
    is good but training Dice is low.
    """
    n_sn = raw_nf.shape[0]
    n_edges = raw_ei.shape[1]

    inv = node_invariants(raw_nf, C)
    int_std = compute_intensity_std(labels_np, ct_u8)
    if len(int_std) < n_sn:
        int_std = np.pad(int_std, (0, n_sn - len(int_std)))
    int_std = int_std[:n_sn]

    principal = inv["principal"].astype(np.float32)
    # Eigenvectors have arbitrary sign. Canonicalize sign so the largest absolute
    # component is positive; this reduces random sign flips across cases.
    if len(principal):
        max_comp = np.argmax(np.abs(principal), axis=1)
        signs = np.sign(principal[np.arange(len(principal)), max_comp])
        signs[signs == 0] = 1
        principal = principal * signs[:, None]

    x = np.column_stack([
        np.log1p(inv["V"]),             # volume
        np.log1p(inv["surface"]),       # boundary/surface
        inv["compactness"],             # compactness
        inv["elongation"],              # elongation
        principal[:, 0],                 # dominant axis x
        principal[:, 1],                 # dominant axis y
        principal[:, 2],                 # dominant axis z
        inv["chan"][:, 0],              # mean intensity
        int_std,                         # intensity std
    ]).astype(np.float32)

    # Per-graph z-score normalization.
    for col in range(x.shape[1]):
        mu, sigma = float(x[:, col].mean()), float(x[:, col].std())
        if sigma > 1e-8:
            x[:, col] = (x[:, col] - mu) / sigma
        else:
            x[:, col] = 0.0

    if n_edges > 0:
        edge_attr = edge_invariants(raw_nf, raw_ei, raw_ef, C)
        ei_fwd = torch.tensor(raw_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr_t = torch.tensor(np.concatenate([edge_attr, edge_attr]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr_t = torch.zeros((0, 10), dtype=torch.float32)

    flat = labels_np.ravel()
    valid = flat >= 0
    gt = (seg.ravel() == 2).astype(np.float64)
    max_id = int(flat[valid].max()) if valid.any() else -1
    y = np.zeros(n_sn, dtype=np.int64)
    if max_id >= 0:
        tumor_count = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1)
        total_count = np.bincount(flat[valid], minlength=max_id + 1)
        overlap = tumor_count / np.maximum(total_count, 1)
        y[:min(n_sn, len(overlap))] = (overlap[:n_sn] >= overlap_threshold).astype(np.int64)

    return Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=edge_index,
        edge_attr=edge_attr_t,
        y=torch.tensor(y, dtype=torch.long),
    )


# Build graphs for all volumes using selected Θopt.
graphs = {}
raw_data = {}
oracles, sns, del_pcts = [], [], []

print(f"Building graphs with Θopt: ψ={MERGE_DIST}, α={CUT_DIST}, βmin={DELETE_SMALL}, βmax=n^{DELETE_LARGE_FRAC:.2f}, m=[{VALUE_MIN},{VALUE_MAX}]")
for vid in ordered:
    ct_raw, seg, ct_u8 = load_and_convert(vid)
    n_vox = ct_raw.size
    params = dict(psi=MERGE_DIST, alpha=CUT_DIST, beta_min=DELETE_SMALL,
                  beta_max_frac=DELETE_LARGE_FRAC, m_min=VALUE_MIN, m_max=VALUE_MAX)
    raw_nf, raw_ei, raw_ef, raw_labels, raw_adj = run_minor(ct_u8, n_vox, params)
    labels_np = np.asarray(raw_labels)

    data = build_pyg_graph(raw_nf, raw_ei, raw_ef, labels_np, seg, ct_u8, overlap_threshold=OVERLAP_THRESHOLD)
    graphs[vid] = data
    raw_data[vid] = {"labels": labels_np, "seg": seg, "n_sn": raw_nf.shape[0]}

    od, t_sn, del_pct = oracle_dice(labels_np, seg)
    oracles.append(od); sns.append(raw_nf.shape[0]); del_pcts.append(del_pct)
    n_tu = int((data.y == 1).sum())
    n_bg = int((data.y == 0).sum())
    print(f"vol-{vid}: {data.num_nodes:,} nodes ({n_tu} tumor, {n_bg:,} bg), "
          f"oracle={od:.4f}, del_tumor={del_pct:.1f}%, edges={data.num_edges:,}, edge_dim={data.edge_attr.shape[1] if data.edge_attr.numel() > 0 else 0}")

mean_oracle = float(np.mean(oracles))
mean_sn = float(np.mean(sns))
mean_del = float(np.mean(del_pcts))
print("\nGraph diagnostics:")
print(f"  Oracle Dice:    {mean_oracle:.4f} ± {np.std(oracles):.4f}")
print(f"  Mean supernodes:{mean_sn:,.0f} ± {np.std(sns):,.0f}")
print(f"  Tumor deleted:  {mean_del:.2f}%")
print("  Paper LiTS target: ~1,075 ± 297 supernodes")

if mean_oracle < 0.80:
    print("\nWARNING: Oracle is low. The coarsener is the bottleneck; GINE training cannot reach 0.89 until this improves.")


## 6. GINE Training

Paper spec: 3-layer GINE, hidden 128, Adam lr=1e-3, patience 10.
Luke's additions: sqrt class weights (capped at 30), patience 30.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, BatchNorm


class GINE(nn.Module):
    """3-layer GINE — paper spec, with dynamic node/edge dimensions."""
    def __init__(self, node_dim, edge_dim, hidden=128):
        super().__init__()
        self.edge_proj = nn.Linear(edge_dim, hidden)

        def mlp(d_in):
            return nn.Sequential(
                nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                nn.Linear(hidden, hidden),
            )

        self.conv1 = GINEConv(mlp(node_dim), edge_dim=hidden)
        self.bn1 = BatchNorm(hidden)
        self.conv2 = GINEConv(mlp(hidden), edge_dim=hidden)
        self.bn2 = BatchNorm(hidden)
        self.conv3 = GINEConv(mlp(hidden), edge_dim=hidden)
        self.bn3 = BatchNorm(hidden)
        self.head = nn.Linear(hidden, 2)

    def forward(self, data):
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        if ea is not None and ea.numel() > 0:
            ea = self.edge_proj(ea)
        else:
            n = x.size(0)
            ei = torch.stack([torch.arange(n, device=x.device)] * 2)
            ea = torch.zeros(n, self.edge_proj.out_features, device=x.device)
        x = F.relu(self.bn1(self.conv1(x, ei, ea)))
        x = F.relu(self.bn2(self.conv2(x, ei, ea)))
        x = F.relu(self.bn3(self.conv3(x, ei, ea)))
        return self.head(x)


In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

train_graphs = [graphs[v] for v in train_ids]
val_graphs = [graphs[v] for v in val_ids]

node_dim = train_graphs[0].x.shape[1]
sample_ea = train_graphs[0].edge_attr
edge_dim = sample_ea.shape[1] if sample_ea.numel() > 0 else 10
print(f"Node dim: {node_dim}; edge dim: {edge_dim}")

model = GINE(node_dim=node_dim, edge_dim=edge_dim).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

total_pos = sum(int((g.y == 1).sum()) for g in train_graphs)
total_neg = sum(int((g.y == 0).sum()) for g in train_graphs)
raw_ratio = total_neg / max(total_pos, 1)
eff_ratio = min(np.sqrt(raw_ratio), 30.0)
weight = torch.tensor([1.0, eff_ratio], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weight)
print(f"Class weight: [1.0, {eff_ratio:.1f}] (raw ratio: {raw_ratio:.1f})")
print(f"Tumor SN: {total_pos:,}; Background SN: {total_neg:,}")


def lifted_voxel_dice_for_vids(model, vids, device):
    """Compute dataset-level lifted voxel Dice."""
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for vid in vids:
            data = graphs[vid]
            rd = raw_data[vid]
            logits = model(data.to(device))
            preds = logits.argmax(dim=1).cpu().numpy()
            labels_np = rd["labels"]
            seg = rd["seg"]
            flat = labels_np.ravel()
            valid = flat >= 0
            pred_mask = np.zeros(labels_np.shape, dtype=bool)
            if valid.any():
                max_id = int(flat[valid].max())
                lut = np.zeros(max_id + 1, dtype=np.int8)
                lut[:min(len(preds), max_id + 1)] = preds[:min(len(preds), max_id + 1)]
                pred_mask = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
            gt_mask = seg == 2
            inter = int((pred_mask & gt_mask).sum())
            tp += inter
            fp += int(pred_mask.sum()) - inter
            fn += int(gt_mask.sum()) - inter
    return 2 * tp / (2 * tp + fp + fn + 1e-8)


PATIENCE = 30
best_dice, best_state, wait = -1.0, None, 0
history = {"train_loss": [], "val_voxel_dice": []}

for epoch in range(1, 201):
    model.train()
    total_loss = 0.0
    for idx in np.random.permutation(len(train_graphs)):
        g = train_graphs[idx].to(device)
        opt.zero_grad()
        loss = criterion(model(g), g.y)
        loss.backward()
        opt.step()
        total_loss += float(loss.item())
    mean_loss = total_loss / max(len(train_graphs), 1)
    history["train_loss"].append(mean_loss)

    val_dice = lifted_voxel_dice_for_vids(model, val_ids, device)
    history["val_voxel_dice"].append(val_dice)

    if val_dice > best_dice:
        best_dice = val_dice
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stop epoch {epoch}, best lifted val Dice={best_dice:.4f}")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  loss={mean_loss:.4f}  lifted_val_dice={val_dice:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest lifted voxel val Dice: {best_dice:.4f}")


## 7. Voxel-Level Evaluation

Lift supernode predictions to voxel grid via the label volume. Each voxel inherits its supernode's prediction.

In [ ]:
model.eval()
model = model.to(device)
results = []

for vid in ordered:
    data = graphs[vid]
    rd = raw_data[vid]
    labels_np = rd["labels"]
    seg = rd["seg"]

    with torch.no_grad():
        preds = model(data.to(device)).argmax(dim=1).cpu().numpy()

    flat = labels_np.ravel()
    valid = flat >= 0
    pred_mask = np.zeros(labels_np.shape, dtype=bool)
    if valid.any():
        max_id = int(flat[valid].max())
        lut = np.zeros(max_id + 1, dtype=np.int8)
        lut[:min(len(preds), max_id + 1)] = preds[:min(len(preds), max_id + 1)]
        pred_mask = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)

    gt_mask = seg == 2
    inter = int((gt_mask & pred_mask).sum())
    dice = 2.0 * inter / (gt_mask.sum() + pred_mask.sum() + 1e-8)
    recall = inter / (gt_mask.sum() + 1e-8)
    precision = inter / (pred_mask.sum() + 1e-8) if pred_mask.sum() > 0 else 0.0

    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    results.append({"vid": vid, "split": split, "dice": dice, "recall": recall, "precision": precision})
    print(f"vol-{vid} [{split}]: Dice={dice:.4f}  Recall={recall:.4f}  Prec={precision:.4f}")

print("\n" + "=" * 60)
for split in ["train", "val", "test"]:
    scores = [r["dice"] for r in results if r["split"] == split]
    if scores:
        print(f"{split:>5s}: Dice = {np.mean(scores):.4f} ± {np.std(scores):.4f}  (n={len(scores)})")

print(f"\nOracle Dice:     {mean_oracle:.4f}")
print(f"Mean supernodes: {mean_sn:.0f}")
print(f"Tumor deleted:   {mean_del:.2f}%")
print("Paper target:    Dice 0.891 ± 0.007, ~1075 SN")

# Save a compact run summary for comparison across parameter-search runs.
summary = {
    "theta": {"psi": MERGE_DIST, "alpha": CUT_DIST, "beta_min": DELETE_SMALL,
              "beta_max_frac": DELETE_LARGE_FRAC, "m_min": VALUE_MIN, "m_max": VALUE_MAX},
    "mean_oracle": mean_oracle,
    "mean_supernodes": mean_sn,
    "mean_tumor_deleted_pct": mean_del,
    "results": results,
}
with open("semir_lits_v3_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved semir_lits_v3_run_summary.json")


## 8. How to interpret this run

The main diagnostic is still the oracle Dice.

- If `Oracle Dice < 0.80`, the coarsener is still the bottleneck; increase `N_RANDOM`, expand the search ranges, or test the HU `[0, 200]` window.
- If `Oracle Dice > 0.90` but test Dice is low, focus on training, label threshold, class weighting, and features.
- If graph size is far above the paper's `~1,075` LiTS supernodes, broaden the search toward larger `ψ` and larger `α`, but make sure oracle Dice does not collapse.

The notebook saves `semir_lits_v3_run_summary.json` so you can compare runs without scrolling through output.
